# District 1: Missing RT https://github.com/cal-itp/gtfs-curator/issues/82

In [1]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

# from shared_utils import bq_utils
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data
from update_vars import DIGEST_DICT

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-pack

In [2]:
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, abbrev_month, analysis_month 

In [3]:
@cache
def gcs_pandas():
    return GCSPandas()

In [4]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [5]:
GCS_PATH = f"{GTFS_DATA_DICT.dir}processed/"

In [6]:
f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [7]:
raw_url = "gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_09.parquet"

In [8]:
fct_monthly_operator_df = pd.read_parquet(raw_url,
                                        storage_options={"token": credentials.token}).reset_index()

In [9]:
fct_monthly_operator_df.sample()

,index,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
1709,1709,10,2026,2026-10-01,Victor Valley Schedule,None,None,None,None,None,Saturday,1578,526.00,1002.20,334.10,0.00,0.00,29,64,1019,38685,12895.00,3,1,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [10]:
fct_monthly_operator_df.vp_name.unique()

array([None, 'TCRTA Vehicle Positions', 'SLORTA VehiclePositions',
       'Tahoe Transportation District GMV VehiclePositions',
       'Merced VehiclePositions',
       'Antelope Valley Transit Authority Vehicle Positions',
       'Union City Vehicle Positions', 'OmniTrans Vehicle Positions',
       'eTrans VehiclePositions', 'Bear Vehicle Positions',
       'WeHo Vehicle Positions',
       'Bay Area 511 Tri Delta VehiclePositions',
       'Bay Area 511 AC Transit Vehicle Position',
       'Bay Area 511 Capitol Corridor VehiclePositions',
       'Bay Area 511 County Connection VehiclePositions',
       'Bay Area 511 ACE VehiclePositions',
       'Bay Area 511 Caltrain VehiclePositions',
       'Bay Area 511 Dumbarton Express VehiclePositions',
       'Bay Area 511 Emery Go-Round VehiclePositions',
       'Bay Area 511 Fairfield and Suisun Transit Vehicle Positions',
       'Bay Area 511 Golden Gate Transit Vehicle Positions',
       'Bay Area 511 Marin VehiclePositions',
       'Bay Ar

In [11]:
fct_monthly_operator_df.tu_name.unique()

array([None, 'TCRTA Trip Updates', 'SLORTA TripUpdates',
       'Tahoe Transportation District GMV TripUpdates',
       'Merced Trip Updates',
       'Antelope Valley Transit Authority Trip Updates',
       'BruinBus Trip Updates', 'Union City Trip Updates',
       'OmniTrans Trip Updates', 'eTrans TripUpdates',
       'Bear Trip Updates', 'WeHo Trip Updates',
       'Bay Area 511 Tri Delta TripUpdates',
       'Bay Area 511 AC Transit Trip Updates',
       'Bay Area 511 Capitol Corridor TripUpdates',
       'Bay Area 511 BART TripUpdates',
       'Bay Area 511 County Connection TripUpdates',
       'Bay Area 511 ACE TripUpdates',
       'Bay Area 511 Caltrain TripUpdates',
       'Bay Area 511 Dumbarton Express TripUpdates',
       'Bay Area 511 Emery Go-Round TripUpdates',
       'Bay Area 511 Fairfield and Suisun Transit TripUpdates',
       'Bay Area 511 Golden Gate Trip Updates',
       'Bay Area 511 Marin TripUpdates',
       'Bay Area 511 Mountain View Community Shuttle TripUpda

In [12]:
fct_monthly_operator_df = fct_monthly_operator_df[['schedule_name','vp_name','month_first_day','tu_name','n_days_schedule_only',
       'n_days_schedule_and_rt', 'n_days_schedule_and_vp_only',
       'n_days_schedule_and_tu_only',]].fillna("N/A")

In [13]:
(fct_monthly_operator_df.loc[fct_monthly_operator_df.tu_name.str.contains('Lake')]
    [['schedule_name','vp_name','month_first_day','tu_name','n_days_schedule_only',
       'n_days_schedule_and_rt', 'n_days_schedule_and_vp_only',
       'n_days_schedule_and_tu_only',]].drop_duplicates())

,schedule_name,vp_name,month_first_day,tu_name,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only


In [14]:
(fct_monthly_operator_df.loc[fct_monthly_operator_df.schedule_name.isin(['Lake Schedule'])]
    [['schedule_name','vp_name','month_first_day','tu_name','n_days_schedule_only',
       'n_days_schedule_and_rt', 'n_days_schedule_and_vp_only',
       'n_days_schedule_and_tu_only',]].drop_duplicates())

,schedule_name,vp_name,month_first_day,tu_name,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only
100,Lake Schedule,N/A,2026-09-01,N/A,N/A,N/A,N/A,N/A
509,Lake Schedule,N/A,2026-08-01,N/A,N/A,N/A,N/A,N/A
543,Lake Schedule,N/A,2026-10-01,N/A,N/A,N/A,N/A,N/A


In [15]:
fct_monthly_operator_df2 = (fct_monthly_operator_df.loc[fct_monthly_operator_df.tu_name.isin([ 'Redding Trip Updates','Humboldt Trip Updates','Lake Schedule'])]
    [['month_first_day','tu_name','n_days_schedule_only',
       'n_days_schedule_and_rt', 'n_days_schedule_and_vp_only',
       'n_days_schedule_and_tu_only',]].drop_duplicates())

In [16]:
fct_monthly_operator_df2

,month_first_day,tu_name,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only
2255,2026-08-01,Redding Trip Updates,0.00,5.00,0.00,0.00
2256,2026-10-01,Redding Trip Updates,0.00,3.00,0.00,0.00
2257,2026-09-01,Redding Trip Updates,0.00,4.00,0.00,0.00
2261,2026-10-01,Redding Trip Updates,0.00,14.00,0.00,0.00
2262,2026-09-01,Redding Trip Updates,0.00,22.00,0.00,0.00
2263,2026-08-01,Redding Trip Updates,0.00,21.00,0.00,0.00
2270,2026-08-01,Humboldt Trip Updates,0.00,5.00,0.00,0.00
2271,2026-10-01,Humboldt Trip Updates,0.00,3.00,0.00,0.00
2272,2026-09-01,Humboldt Trip Updates,0.00,4.00,0.00,0.00
2273,2026-10-01,Humboldt Trip Updates,0.00,15.00,0.00,0.00


In [17]:
f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [18]:
from update_vars import DIGEST_DICT, PROCESSED_GCS, RAW_GCS, SHARED_GCS, abbrev_month

In [19]:
monthly_routes = gpd.read_parquet(
        f"{RAW_GCS}{DIGEST_DICT.route_map}_{abbrev_month}.parquet",
        storage_options={"token": credentials.token},
    )

In [20]:
monthly_routes.columns

Index(['name', 'year', 'month', 'month_first_day', 'route_name',
       'direction_id', 'route_type', 'shape_id', 'shape_array_key', 'n_trips',
       'geometry'],
      dtype='object')

In [21]:
monthly_routes.month_first_day.unique()

array([datetime.date(2026, 10, 1), datetime.date(2026, 9, 1)],
      dtype=object)